# Production-Grade Prompting, Agents & Tool Use

Welcome back! This is the **Module 2 lab**, the sequel to *MSO Foundations*.

Here's the idea that drives this whole notebook: writing code that *uses* Claude is a very different job from using Claude to *write* code. A prototype that looks great on your laptop can fall over the moment real users, real documents and real traffic show up. So we'll take a small contract-review prototype and harden it, one failure at a time.

Each chapter picks **one production failure** and shows you the fix:

| # | Chapter | Production failure it prevents |
|---|---|---|
| 1 | Setup | — |
| 2 | Prompting craft | Output in the wrong shape breaks the parser |
| 3 | Extended thinking | Paying for reasoning you don't need |
| 4 | Tool use & schema design | Claude picks the wrong tool; the loop breaks |
| 5 | Streaming | A dropped stream corrupts history |
| 6 | Context engineering | The context window fills at turn 8 |
| 7 | Agent construction | An agent makes an irreversible change unchecked |
| 8 | Agent memory | State is lost (or bloats every call) between sessions |
| 9 | Multimodal & batch | Image costs and rate limits blow up at scale |
| 10 | Debug challenge | All of the above in one buggy agent |

Let's get started 🚀

### Prerequisite
Please download [AWS1.pdf](https://drive.google.com/file/d/1XSe2pSsGN1ssAbif92rvb80AnHb_Ni0F/view?usp=sharing) (the short AWS contract from the MSO lab) before you begin. You'll upload it at the end of Chapter 1.

## Chapter 1 — Setup

Let's get the boring-but-necessary part out of the way first. We'll install four packages:

1. **`anthropic`**: the official Python SDK for talking to Claude.
2. **`PyMuPDF`** (`fitz`): reads the text out of our PDF contract.
3. **`pyboxen`**: draws neat titled boxes around output so results are easy to spot as you scroll.
4. **`pillow`**: lets us create test images in Chapter 9.

#### What are we going to do?

Run one install command. It takes a few seconds, and `-q` keeps the output quiet.

In [ ]:
!pip install -q -U anthropic PyMuPDF pyboxen pillow

Next up, the imports and a few small helpers that we'll reuse in every chapter.

#### What are we going to do?

1. Import the libraries we just installed.
2. Ask for your Anthropic API key with `getpass`, so it's typed in privately and never saved in the notebook.
3. Create the `client` we'll use to call Claude, and name the two models we'll use: **Haiku** (fast and cheap) and **Sonnet** (more capable).
4. Define two tiny helpers: `show()` prints output in a titled, colored box, and `text_of()` pulls the plain text out of Claude's response.

When you run the cell, a prompt will appear. Paste your key there.

In [ ]:
import os, json, math, base64, asyncio, time
from getpass import getpass

import anthropic
import fitz  # PyMuPDF
from pyboxen import boxen

os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")
client = anthropic.Anthropic()

HAIKU, SONNET = "claude-haiku-4-5", "claude-sonnet-4-6"


def show(title, text, color="cyan"):
    print(boxen(str(text), title=title, color=color, padding=1))


def text_of(response):
    """Join all text blocks of a response."""
    return "".join(b.text for b in response.content if b.type == "text")

Now let's load the contract. We'll reuse it in almost every chapter, so it's worth doing once, properly.

#### What are we going to do?

1. Open the file picker so you can upload `AWS1.pdf`.
2. Read the PDF **page by page** and keep each page's text in a list called `pages`. Later, our tools will need to fetch individual pages.
3. Join all the pages into one string, `contract_text`, for the times we want to hand Claude the whole contract.
4. Print a quick sanity check: how many pages, and how many characters.

In [ ]:
from google.colab import files
files.upload()

with fitz.open("/content/AWS1.pdf") as doc:
    pages = [p.get_text() for p in doc]
contract_text = "\n".join(pages)
print(f"{len(pages)} pages · {len(contract_text):,} characters")

## Chapter 2 — Prompting Craft

You've probably had this happen: a prompt works perfectly in the chat window, then breaks in production on an input you never tested. The natural reaction is to pile on more words. Please don't! Instead, **work out which structural piece is missing**:

| What you observe | What's missing | Fix |
|---|---|---|
| Wrong *shape* (a sentence instead of a label) | **Output constraint** | State the exact form, the allowed values, and "nothing else" |
| Scope or tone drifts across turns | **System prompt** | A persistent role plus rules |
| Right task, but Claude invents its own structure | **Few-shot examples** | Show one correct input → output pair |
| Instructions blend into the data | **XML tags** | Use `<clause>…</clause>` boundaries |

**Our task:** imagine a router that needs exactly one of four labels for every contract clause. If Claude replies with a chatty sentence, the router chokes. So instead of eyeballing outputs, we'll *measure* how often the output is usable: one valid label, nothing else.

#### What are we going to do?

1. Define the four allowed labels and five sample clauses (one of them is deliberately ambiguous).
2. Write a helper, `valid_rate()`, that classifies every clause three times and reports the share of valid outputs.
3. Run it with a **bare prompt** ("Classify the clause") to get our baseline. Expect it to be bad. That's the point.

In [ ]:
LABELS = {"TERMINATION", "PAYMENT", "LIABILITY", "OTHER"}

clauses = [
    "Either party may terminate this Agreement for convenience on 30 days' written notice.",
    "Customer shall pay all fees within 30 days of the invoice date.",
    "Neither party's aggregate liability shall exceed the fees paid in the prior 12 months.",
    "This Agreement is governed by the laws of the State of Washington.",
    "We may suspend your access if you breach this Agreement, and fees continue to accrue during suspension.",
]


def valid_rate(system, wrap, runs=3):
    """Classify every clause `runs` times; return (share of valid outputs, sample outputs)."""
    outs = []
    for clause in clauses:
        for _ in range(runs):
            r = client.messages.create(
                model=HAIKU, max_tokens=60, system=system,
                messages=[{"role": "user", "content": wrap(clause)}],
            )
            outs.append(text_of(r).strip())
    ok = sum(o in LABELS for o in outs)
    return ok / len(outs), outs


bare_rate, bare_outs = valid_rate("You are a clause classifier. Classify the clause.", lambda c: c)
show("❌ Bare prompt", f"Valid outputs: {bare_rate:.0%}\n\nSamples:\n" + "\n".join("• " + o[:80] for o in bare_outs[:4]), "red")

Now let's fix what we just saw. We won't add more words. We'll add *structure*.

#### What are we going to do?

We'll rewrite the prompt with three things, then re-run the exact same measurement:

1. A **system prompt** that sets the role and the rules.
2. An **output constraint**: "Return only the label. No other text."
3. **Few-shot examples** wrapped in XML tags, including the ambiguous "suspend service" case, so Claude sees how to handle it.

Compare the new valid-output rate with the baseline above.

In [ ]:
SYSTEM = """You are a contract clause classifier.
Classify each clause into exactly one of: TERMINATION, PAYMENT, LIABILITY, OTHER.
Return only the label. No other text.

<sample_input>Either party may end this agreement with 60 days' notice.</sample_input>
<ideal_output>TERMINATION</ideal_output>
<sample_input>Invoices are due net 45.</sample_input>
<ideal_output>PAYMENT</ideal_output>
<sample_input>We may suspend service for breach, and fees still accrue.</sample_input>
<ideal_output>TERMINATION</ideal_output>"""

good_rate, good_outs = valid_rate(SYSTEM, lambda c: f"<clause>{c}</clause>")
show("✅ Constrained prompt", f"Valid outputs: {good_rate:.0%}\n\nSamples: {sorted(set(good_outs))}", "green")

### Move the contract into the API: structured outputs

A prompt is a *request*. Claude can still ignore it. A **JSON schema** is *enforced*: the API uses constrained decoding, so tokens that would break the schema simply can't be generated.

- `output_config.format` constrains the **final response**.
- `strict: true` on a tool constrains the **tool arguments** (we'll use this in Chapter 4).

A few honest trade-offs: the first call with a new schema is slower (the grammar gets compiled, then cached for 24 hours), you pay a few extra input tokens, and you still have to check `stop_reason`, because a `refusal` or `max_tokens` can still give you output that doesn't match. Structured outputs also don't work with message prefilling.

#### What are we going to do?

1. Define a schema: a `category` that must be one of our four labels, plus a one-line `summary`.
2. Ask Claude to classify and summarize the tricky "suspend your access" clause, with the schema attached.
3. Check `stop_reason` *before* parsing. This is the habit to build.
4. Parse the JSON and display it.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": sorted(LABELS)},
        "summary": {"type": "string"},
    },
    "required": ["category", "summary"],
    "additionalProperties": False,
}

r = client.messages.create(
    model=SONNET, max_tokens=300,
    output_config={"format": {"type": "json_schema", "schema": schema}},
    messages=[{"role": "user", "content": f"Classify and summarize in one sentence: <clause>{clauses[4]}</clause>"}],
)

assert r.stop_reason == "end_turn", f"Don't parse: stop_reason={r.stop_reason}"  # always check
show("🧱 Schema-enforced output", json.loads(text_of(r)), "green")

> **Rule of thumb:** name the failure, add the one technique that matches it, then re-run. If three re-prompts in a row didn't help, stop adding text. And remember to keep things simple: a "summarize this paragraph" prompt doesn't need any of this.

## Chapter 3 — Extended Thinking

Prompting shapes *what* Claude produces. **Thinking** shapes *how much work Claude does before it answers*. On current models thinking is **adaptive**: Claude decides how much to think, and you steer the depth with `effort`. (The older `budget_tokens` setting is deprecated and errors on the newest models.)

| Task | Thinking? |
|---|---|
| Multi-step reasoning, planning dependent actions | ✅ On, with effort matched to difficulty |
| Classification, extraction, format conversion | ❌ Off. A constrained prompt is cheaper and just as accurate |
| Agent loops with tools | ✅ On, but follow the **carry-back rule** below |

Thinking tokens are billed like output tokens, so it's worth seeing the cost with your own eyes.

#### What are we going to do?

1. Write one genuinely hard question: *every way the contract can end or access can be suspended, who triggers each, and which is worst for the customer*.
2. Ask it twice on Sonnet: once with thinking **off**, once with **adaptive thinking at high effort**.
3. Print the block types, the output token count and the start of each answer, so you can compare quality against cost.

In [ ]:
hard_q = (
    "Using the contract below, list every way the contract can end or access can be suspended, "
    "who can trigger each, and which one gives the customer the least protection. Cite sections.\n\n" + contract_text
)

runs = {
    "thinking OFF": dict(thinking={"type": "disabled"}),
    "adaptive · effort=high": dict(
        thinking={"type": "adaptive", "display": "summarized"}, output_config={"effort": "high"}
    ),
}

for name, kwargs in runs.items():
    r = client.messages.create(model=SONNET, max_tokens=6000,
                               messages=[{"role": "user", "content": hard_q}], **kwargs)
    show(f"{name} · blocks={[b.type for b in r.content]} · output_tokens={r.usage.output_tokens}",
         text_of(r)[:700] + " …", "yellow" if "OFF" in name else "purple")

### The carry-back rule

When thinking is on **and** you're using tools, every `thinking` block has to go back to the API **exactly as you received it** on the next turn. It carries a cryptographic signature, so if you edit it, summarize it or drop it, the request gets rejected. The same goes for redacted thinking blocks.

> Worried that reasoning will bloat your context? Don't strip the blocks. Manage the context instead (we'll do that in Chapter 6).

In our loops we'll just append `response.content` untouched, which satisfies the rule automatically.

## Chapter 4 — Tool Use & Schema Design

Here's the thing that surprises most people: **Claude never runs your tools.** It reads your schemas, decides which one fits, and hands back a `tool_use` block. *Your code* runs the tool and sends the answer back as a `tool_result`.

```text
define schema → send message → tool_use block → YOU execute → tool_result → Claude continues
```

| Block | Rule |
|---|---|
| `text` | Keep it when you append the assistant turn |
| `tool_use` | Has a unique `id` |
| `tool_result` | Must be in the **very next user turn**, with the same `tool_use_id` (set `is_error: true` on failure) |
| `thinking` | Pass it back unchanged |

A schema has a `name`, a `description` and an `input_schema`. **The description is what Claude uses to choose a tool**, so write when to use it *and when not to*.

#### What are we going to do?

We'll give Claude two real tools over our contract:

1. **`search_contract`** finds a keyword and returns short snippets with page numbers.
2. **`get_page`** returns the text of a single page (capped, to protect our context).

We'll register them in a lookup table (`IMPL`) so our code can run whichever one Claude asks for. Then we'll define two versions of the tool schemas: `tools_good` with specific descriptions, and `tools_bad` with the same tools but a vague, overlapping description. The next few cells put them head to head.

In [ ]:
def search_contract(keyword):
    """Return up to 5 short snippets (with page numbers) that contain the keyword."""
    hits = []
    for n, text in enumerate(pages, 1):
        i = text.lower().find(keyword.lower())
        if i >= 0:
            hits.append(f"[page {n}] …{text[max(0, i - 120): i + 200]}…".replace("\n", " "))
    return "\n".join(hits[:5]) or "No matches."


def get_page(page):
    """Return the text of one page (capped at 2,500 characters to protect context)."""
    if not 1 <= page <= len(pages):
        raise ValueError(f"page must be 1–{len(pages)}")
    return pages[page - 1][:2500]


IMPL = {"search_contract": search_contract, "get_page": get_page}

tools_good = [
    {
        "name": "search_contract",
        "description": (
            "Search the contract for a keyword and return short snippets with page numbers. "
            "Use this when you don't yet know where a topic appears. "
            "Do not use this if you already know the page number — use get_page instead."
        ),
        "input_schema": {"type": "object", "properties": {"keyword": {"type": "string"}}, "required": ["keyword"]},
    },
    {
        "name": "get_page",
        "description": (
            "Return the full text of one contract page. Use this only when you already know the page number "
            "(for example from a search result). Do not use this to look for a topic."
        ),
        "input_schema": {"type": "object", "properties": {"page": {"type": "integer"}}, "required": ["page"]},
    },
]

# Same tools with overlapping, vague descriptions — Claude can't tell them apart
tools_bad = [
    {**t, "description": "Use this to find information in the contract."} for t in tools_good
]

**Which tool does Claude pick first?** Let's find out. We'll ask the same questions with vague descriptions and with specific ones. (Vague descriptions don't fail every single time. They fail on boundary cases, and they get worse as you add more tools.)

#### What are we going to do?

1. Write a small helper that sends a question and returns the *name* of the first tool Claude chooses.
2. Ask two questions: one that needs a search ("Where does the contract discuss termination?") and one that names a page ("Show me everything on page 3.").
3. Run both against `tools_bad` and `tools_good`, then compare what Claude picked with what we expected.

In [ ]:
def first_tool(tools, question):
    r = client.messages.create(model=HAIKU, max_tokens=300, tools=tools,
                               messages=[{"role": "user", "content": question}])
    return next((b.name for b in r.content if b.type == "tool_use"), "none")


tests = [("Where does the contract discuss termination?", "search_contract"),
         ("Show me everything on page 3.", "get_page")]

for label, tools in [("vague", tools_bad), ("specific", tools_good)]:
    picks = [(first_tool(tools, q), expected) for q, expected in tests]
    show(f"{label} descriptions", "\n".join(f"picked {p:16} expected {e}" for p, e in picks),
         "red" if label == "vague" else "green")

### The tool-use loop

So far Claude has only *chosen* a tool. Now let's actually close the loop. The pattern is simple: keep running tools until `stop_reason` is no longer `"tool_use"`. Claude may call several tools **in parallel** in a single turn, and all of those results must go back together in **one** user message.

#### What are we going to do?

1. Write `run_tools()`, a loop that calls Claude, keeps the full response in history, runs any requested tools, and sends all results back in one turn.
2. Make failures safe: if a tool raises an error, we pass it back to Claude as an error result instead of crashing.
3. Add a `max_iters` cap so the loop can never run forever.
4. Ask a real question: *what notice period applies to termination for convenience, and on which page?* You'll see each 🔧 tool call print as it happens.

In [ ]:
def run_tools(question, tools=tools_good, max_iters=6):
    messages = [{"role": "user", "content": question}]
    for _ in range(max_iters):
        r = client.messages.create(model=SONNET, max_tokens=1500, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": r.content})   # keep ALL blocks, unchanged
        if r.stop_reason != "tool_use":
            return text_of(r), messages
        results = []
        for b in r.content:
            if b.type != "tool_use":
                continue
            try:
                out, err = IMPL[b.name](**b.input), False
            except Exception as e:                                       # report failures to Claude
                out, err = f"Error: {e}", True
            print(f"🔧 {b.name}({b.input}){'  ❌' if err else ''}")
            results.append({"type": "tool_result", "tool_use_id": b.id, "content": str(out), "is_error": err})
        messages.append({"role": "user", "content": results})        # ALL results, one user turn
    raise RuntimeError("max_iters reached")


answer, history = run_tools("What notice period applies to termination for convenience, and on which page is it?")
show("Answer", answer, "green")

### What a broken pairing looks like

Let's break something on purpose, because it's much nicer to meet this error here than in production.

#### What are we going to do?

1. Take the history from the run above and pull out the assistant turn that contained a `tool_use`.
2. Build a new conversation where the `tool_result` carries the **wrong id** (`toolu_WRONG`).
3. Send it and catch the error.

The API will reject it. No amount of prompt tweaking can fix a structural mistake like this.

In [ ]:
first_assistant = next(m for m in history if m["role"] == "assistant"
                       and any(b.type == "tool_use" for b in m["content"]))
broken = [
    history[0],
    first_assistant,
    {"role": "user", "content": [{"type": "tool_result", "tool_use_id": "toolu_WRONG", "content": "Page text…"}]},
]

try:
    client.messages.create(model=SONNET, max_tokens=300, tools=tools_good, messages=broken)
except anthropic.BadRequestError as e:
    show("🚫 Validation error", e.message, "red")

**Schema design cheat-sheet**

| Decision | Guidance |
|---|---|
| Required fields | Only if the call makes no sense without them. Otherwise Claude invents values |
| Optional fields | Give them sensible defaults. Claude may omit what it doesn't know |
| Description length | 3–4 sentences: what it does, when to use it, when *not* to, and what it returns |
| Dependent calls | Keep them in separate turns. Independent calls run in parallel (`disable_parallel_tool_use` forces one at a time) |
| Near-duplicate tools | Add exclusions, or merge them into one tool with a `type` parameter |
| `strict: true` | Validates the arguments against your schema before your code runs |

One more thing to look forward to: **MCP** (Module 3) lets you *connect* maintained tool servers instead of hand-writing schemas. The loop stays identical, but every connected server adds tool definitions to your context.

## Chapter 5 — Streaming Without Corrupting State

When you stream, Claude doesn't send one finished message. It sends a series of **events**, and *you* assemble the message from them.

| Event | Your handler |
|---|---|
| `message_start` | Create an empty content list |
| `content_block_start` | Open a slot (a `tool_use` block opens with its name and id, but **no input yet**) |
| `content_block_delta` | Append the fragment (tool input arrives as **partial JSON**) |
| `content_block_stop` | The block is complete. This is the **first moment** tool JSON can be parsed |
| `message_delta` | Record the `stop_reason` |
| `message_stop` | The whole message is done. **Only now** do you commit it to history |

> A stream *ending* is not the same as a message *completing*. If the connection drops in the middle of a `tool_use`, the half-built block ends up in your history and breaks the **next** request, and the error points at the wrong place.

#### What are we going to do?

Write `stream_turn()`, a handler that follows those rules. It:

1. Builds each block piece by piece as events arrive, printing text live.
2. Waits for `content_block_stop` before parsing any tool JSON.
3. Returns the content **only if** it saw `message_stop`. Otherwise it reports `interrupted`.
4. Has a `drop_after` switch so we can *simulate* a dropped connection.

We'll start with the healthy case: a normal stream that finishes cleanly.

In [ ]:
def stream_turn(messages, tools, drop_after=None):
    """Stream one turn. Returns (content, stop_reason), or (None, 'interrupted') if the stream never reached message_stop."""
    blocks, stop_reason, finished, seen = {}, None, False, 0
    with client.messages.create(model=SONNET, max_tokens=1000, tools=tools, messages=messages, stream=True) as stream:
        for ev in stream:
            seen += 1
            if ev.type == "content_block_start":
                b = ev.content_block
                blocks[ev.index] = {"type": b.type, "id": getattr(b, "id", None), "name": getattr(b, "name", None), "text": "", "json": ""}
            elif ev.type == "content_block_delta":
                d, blk = ev.delta, blocks[ev.index]
                if d.type == "text_delta":
                    blk["text"] += d.text
                    print(d.text, end="", flush=True)
                elif d.type == "input_json_delta":
                    blk["json"] += d.partial_json            # NOT valid JSON yet
            elif ev.type == "content_block_stop":
                blk = blocks[ev.index]
                if blk["type"] == "tool_use":
                    blk["input"] = json.loads(blk["json"] or "{}")   # safe only now
            elif ev.type == "message_delta":
                stop_reason = ev.delta.stop_reason
            elif ev.type == "message_stop":
                finished = True
            if drop_after and seen >= drop_after:               # simulate a dropped connection
                break
    if not finished:
        return None, "interrupted"
    content = []
    for i in sorted(blocks):
        b = blocks[i]
        content.append({"type": "text", "text": b["text"]} if b["type"] == "text"
                       else {"type": "tool_use", "id": b["id"], "name": b["name"], "input": b["input"]})
    return content, stop_reason


q = [{"role": "user", "content": "Find the termination for convenience clause, then explain it in 2 sentences."}]

print("— healthy stream —")
content, reason = stream_turn(q, tools_good)
print(f"\n\n→ stop_reason={reason}, blocks={[b['type'] for b in content]}")

Now the real test: what happens when the connection drops?

#### What are we going to do?

1. Run `stream_turn()` again, but with `drop_after=8`, so the stream is cut off after just 8 events.
2. Check whether we got a complete turn back. We didn't, so we **discard** the partial turn.
3. Confirm that our history is untouched, which means it's safe to retry.

This is the "gate on `message_stop`" rule in action.

In [ ]:
print("— stream dropped mid-way —")
history2 = list(q)
content, reason = stream_turn(history2, tools_good, drop_after=8)

if content is None:                       # gate on message_stop
    print(f"\n\n→ {reason}: partial turn DISCARDED, history untouched ({len(history2)} message). Safe to retry.")
else:
    history2.append({"role": "assistant", "content": content})

**Handler rules to remember:**

1. Act on a block only after `content_block_stop`.
2. Append to history only after `message_stop`.
3. On interruption, discard the partial turn and retry from the last complete one.
4. Check `stop_reason == "tool_use"` before continuing a loop.

And if your responses are short and backend-only, a plain non-streamed call is simpler and removes this whole class of risk.

## Chapter 6 — Context Engineering

Think of the context window as a **fixed budget**. Everything has to fit inside it: the system prompt, the history, the tool definitions, every tool result, and the answer itself. Tool outputs are the biggest spenders. In production they're typically **3–5× longer** than your test fixtures, so a session that survives 50 turns in dev can hit the ceiling at turn 8. And the symptom looks like *wrong tool selection*, not overflow, which makes it hard to spot.

**A note on model choice:** start with Sonnet. Move up to Opus only if an eval shows Sonnet misses your bar. Move down to Haiku only if an eval shows the drop in quality is acceptable.

Let's measure instead of guessing.

#### What are we going to do?

1. Write `n_tokens()`, which asks the API to count tokens for any text *before* we send it.
2. Set a deliberately small **budget** of 6,000 tokens so we can see the effect quickly.
3. Simulate an agent reading the contract one page per turn, adding up the tokens as it goes.
4. Print a growing bar for each turn, and stop with a warning the moment we blow the budget.

In [ ]:
def n_tokens(text):
    return client.messages.count_tokens(model=SONNET, messages=[{"role": "user", "content": text}]).input_tokens


BUDGET = 6_000          # a deliberately small team-imposed budget, to see the effect
used = n_tokens(SYSTEM)
for turn, page in enumerate(range(1, len(pages) + 1), 1):
    used += n_tokens(get_page(page))
    print(f"turn {turn:2}: +page {page:2} → {used:6,} tokens {'█' * (used // 400)}")
    if used > BUDGET:
        print(f"\n🚨 Budget of {BUDGET:,} exceeded at turn {turn}. Time to prune/compact — before the symptoms start.")
        break

### Four ways to stay in budget

| Strategy | What it does | Use when | You lose |
|---|---|---|---|
| **Pruning** | Rewinds or drops spent tool results | Dead-end path, finished lookups | Work after the rewind point |
| **Compaction** (`/compact`, server-side, or your own summarizer) | Summarizes the history | You're near the ceiling and the same task continues | Anything the summary leaves out |
| **Clearing** (`/clear`, new session) | Empties the context | The next task is unrelated | Everything you didn't persist |
| **Subagent handoff** | Uses an isolated context and returns a summary | Self-contained or exploratory subtasks | Visibility into its steps |

**Compaction is only as good as the summarizer prompt.** Let's prove it.

#### What are we going to do?

1. Take a short transcript with a few traps in it: a file edited by mistake, an error, and a revert.
2. Summarize it twice with Haiku: once with a **generic** prompt, once with a **specific** prompt that says what to preserve (file paths, decisions, errors and how they were resolved).
3. Compare the two summaries. Does the generic one still tell you that `staging.yaml` was reverted?

In [ ]:
transcript = """
user: Please update the rate limit in config/prod.yaml from 100 to 500.
assistant: I edited config/prod.yaml (rate_limit: 500) and config/staging.yaml (rate_limit: 500).
user: validate_config failed: 'burst' must be <= rate_limit.
assistant: Fixed by setting burst: 400 in config/prod.yaml. validate_config now passes.
user: Don't touch staging, that was a mistake. Revert it.
assistant: Reverted config/staging.yaml to rate_limit: 100.
"""

generic = "Summarize the conversation so far.\n" + transcript
specific = ("Summarize the conversation, preserving: all file paths modified, all decisions made, "
            "and any errors encountered and how they were resolved.\n" + transcript)

for name, p in [("generic summarizer", generic), ("specific summarizer", specific)]:
    r = client.messages.create(model=HAIKU, max_tokens=300, messages=[{"role": "user", "content": p}])
    show(name, text_of(r), "red" if name.startswith("generic") else "green")

### Prompt caching: pay once for the stable prefix

If you send the same big prefix on every request, you're paying for it every time. Prompt caching fixes that. Mark the last block you want cached with `cache_control: {"type": "ephemeral"}` (you get up to 4 breakpoints, and the order is tools → system → messages). A few rules to keep in mind:

- The prefix must match **exactly**.
- The default TTL is 5 minutes (`"ttl": "1h"` is opt-in).
- There's a minimum size, around 1,024 tokens on most models. Shorter prompts silently don't cache.

#### What are we going to do?

1. Put the whole contract inside the system prompt and mark it for caching.
2. Ask two different questions about it, back to back.
3. Print the cache usage for each request: how many tokens were **written** to the cache, how many were **read** from it, and how many were billed as normal input.

In [ ]:
system_blocks = [
    {"type": "text", "text": "You answer questions about the contract below.\n\n" + contract_text,
     "cache_control": {"type": "ephemeral"}}
]

for i, q in enumerate(["Who can terminate the agreement?", "What happens to customer content after termination?"], 1):
    r = client.messages.create(model=SONNET, max_tokens=200, system=system_blocks,
                               messages=[{"role": "user", "content": q}])
    u = r.usage
    print(f"request {i}: cache_write={u.cache_creation_input_tokens or 0:>6,}  cache_read={u.cache_read_input_tokens or 0:>6,}  uncached_input={u.input_tokens:>5,}")

Look at the numbers: request 1 **writes** the cache, and request 2 **reads** it at a fraction of the cost. For multi-turn sessions with a stable system prompt and tool schemas, this is the highest-leverage cost cut you'll find. And `count_tokens` (which we used above) lets you stop oversized requests *before* they fail.

One last thing: **subagents** start with a clean slate. They only get the task and whatever context you hand over, and they don't inherit your skills.

## Chapter 7 — Building a Production Agent

An **agent** is a multi-step tool-use loop, plus managed context, plus a goal. That extra power comes with extra coordination cost and more ways to fail, so decide *first* whether you really need one:

| Choose a **workflow** when… | Choose an **agent** when… |
|---|---|
| You can list the exact steps in code | You can specify the goal and the tools, but not the path |
| Error cost is real and step guardrails matter | Inputs vary unpredictably |
| Every run follows the same sequence | The task needs creative sequencing of tools |

> Progression: single API call → workflow → agent. Only move up when the simpler pattern can't cope.

**Who runs the loop?** It's the same pattern with three different wiring options:

| Path | You own | Pick when |
|---|---|---|
| **Raw Messages API loop** | Everything | You want full control, or you're learning the loop |
| **Agent SDK** | Tool execution + your app (the SDK runs the loop in your process) | You want Claude Code's loop; set `settingSources` explicitly |
| **Managed Agents** (beta) | Your app + the agent definition (Anthropic runs the loop and sandbox) | Long-running work. ⚠️ Server-side state means it's not eligible for ZDR / HIPAA BAA |

**Loop checklist:** ① tools registered ② system prompt scoped to the task ③ every `tool_use` answered ④ **a human gate before irreversible actions** ⑤ explicit exit conditions.

**Where does a human gate go?** Ask yourself: *"What's the worst that could happen if this step runs unchecked?"* Put a gate before destructive, write or send calls, after a plan, and on unexpected output.

#### What are we going to do?

We'll build a small contract-review agent that has a dangerous tool. `send_report` "emails" a result, and that can't be undone, so it needs approval.

1. Add `send_report` to our toolset. Its description says it's irreversible and when *not* to use it. (Here it only records the email in a list, so nothing really gets sent.)
2. Mark it as `NEEDS_APPROVAL` and write a `human_approves()` checkpoint that pauses and asks you y/N.
3. Write `run_agent()` with two exit conditions: an iteration cap, and Claude finishing on its own.
4. Give it a goal: find the termination notice rules and email a 3-bullet summary. When the checkpoint box appears, type your answer.

Run it twice: **y** once and **N** once.

In [ ]:
sent_reports = []


def send_report(to, body):
    sent_reports.append((to, body))          # stand-in for a real, irreversible email
    return f"Report sent to {to}."


IMPL["send_report"] = send_report

agent_tools = tools_good + [{
    "name": "send_report",
    "description": (
        "Email a final report to a recipient. This is irreversible. Use it once, only after the analysis "
        "is complete. Do not use it for drafts or intermediate notes."
    ),
    "input_schema": {
        "type": "object",
        "properties": {"to": {"type": "string"}, "body": {"type": "string"}},
        "required": ["to", "body"],
    },
}]

NEEDS_APPROVAL = {"send_report"}


def human_approves(name, args):
    show("✋ Human checkpoint", f"Agent wants to run {name}\n\n{json.dumps(args, indent=2)[:600]}", "yellow")
    return input("Approve? (y/N): ").strip().lower() == "y"


def run_agent(goal, max_iters=10):
    messages = [{"role": "user", "content": goal}]
    system = ("You are a contract-review agent. Tools: search_contract, get_page, send_report. "
              "Research with the read tools, then send exactly one report. Stop when the report is sent.")
    for i in range(max_iters):                                     # exit condition #1: iteration cap
        r = client.messages.create(model=SONNET, max_tokens=1500, system=system, tools=agent_tools, messages=messages)
        messages.append({"role": "assistant", "content": r.content})
        if r.stop_reason != "tool_use":                            # exit condition #2: model is done
            return text_of(r), messages
        results = []
        for b in r.content:
            if b.type != "tool_use":
                continue
            if b.name in NEEDS_APPROVAL and not human_approves(b.name, b.input):
                results.append({"type": "tool_result", "tool_use_id": b.id, "is_error": True,
                                "content": "Rejected by human reviewer. Do not retry; ask what to change."})
                continue
            try:
                out, err = str(IMPL[b.name](**b.input)), False
            except Exception as e:
                out, err = f"Error: {e}", True
            print(f"🔧 [{i}] {b.name}({str(b.input)[:80]})")
            results.append({"type": "tool_result", "tool_use_id": b.id, "content": out, "is_error": err})
        messages.append({"role": "user", "content": results})
    raise RuntimeError("Agent hit max_iters without finishing")


final, _ = run_agent("Find the termination notice rules and email a 3-bullet summary to legal@example.com.")
show("Agent result", final, "green")
print("Reports actually sent:", len(sent_reports))

Did you try both answers? When you said **N**, notice that the rejection went back to Claude as an *error result* rather than crashing the program. Claude can read it, adapt, and ask what to change. That's what a well-behaved gate looks like.

> **Over-tooling** is the most common production bug here: registering every tool "just in case" makes routing worse. Start with the minimum set and add a tool only when you've confirmed a gap.

**Regulated data picks your endpoint *before* you design anything.** HIPAA calls for a BAA-covered configuration (not Console, Workbench or consumer plans). EU residency means Bedrock or Vertex with a pinned region. FedRAMP means Claude for Government, Bedrock GovCloud, or Vertex Assured Workloads. Privilege means the direct API through the firm's logged gateway. (Module 4 goes deeper on security.)

## Chapter 8 — Agent Memory

How much your agent should remember is decided by **the shape of your sessions**, not by whatever is easiest to code:

| Scope | Persists | Cost | Use when |
|---|---|---|---|
| **In-context** | Within one session | Tokens grow every turn | Short, one-off sessions |
| **External storage** | Across sessions and users | Retrieval latency + code | Multi-day threads, shared state |
| **Summarized** | Condensed history | Cheap, but lossy | Long conversations |
| **Stateless** | Nothing | None | Self-contained jobs |

Here's the classic failure: in development you used one long session, but production runs many short ones, and by session 4 the in-context history is already 40k tokens. Refactoring at that point takes hours. Deciding at design time takes minutes.

#### What are we going to do?

We'll build **external storage plus a targeted summarizer** in miniature:

1. Write `save_memory()` and `load_memory()`, which keep one summary per case in a small JSON file.
2. **Session 1:** take a short conversation about case C-17, ask Haiku to summarize it (telling it exactly what to keep: the case id, the concern, decisions and open questions), and save the summary.
3. **Session 2:** start fresh, with no history at all. Load the saved summary into the system prompt and ask Claude what we concluded.

If it recalls Session 1 correctly, memory is working.

In [ ]:
MEMORY_FILE = "/content/agent_memory.json"


def save_memory(case_id, summary):
    mem = json.load(open(MEMORY_FILE)) if os.path.exists(MEMORY_FILE) else {}
    mem[case_id] = summary
    json.dump(mem, open(MEMORY_FILE, "w"), indent=2)


def load_memory(case_id):
    return json.load(open(MEMORY_FILE)).get(case_id, "") if os.path.exists(MEMORY_FILE) else ""


# ── Session 1 (illustrative transcript) ──
s1 = [
    {"role": "user", "content": "Review the AWS termination terms for case C-17. Key concern: data deletion."},
    {"role": "assistant", "content": "After termination the customer has 30 days to retrieve content. Open question: what happens to backups?"},
]
summary = text_of(client.messages.create(model=HAIKU, max_tokens=250, messages=[{"role": "user", "content":
    "Summarize for the next session, preserving: the case id, the user's concern, decisions made, open questions.\n\n"
    + json.dumps(s1)}]))
save_memory("C-17", summary)

# ── Session 2 (fresh process, no history) ──
r = client.messages.create(model=HAIKU, max_tokens=200,
    system="Prior context for this case:\n" + load_memory("C-17"),
    messages=[{"role": "user", "content": "Remind me what we concluded and what is still open."}])
show("Session 2 — recalls Session 1 from storage", text_of(r), "green")

### Skills: reusable instructions that load *on demand*

A **Skill** is a `SKILL.md` file: some frontmatter (`name` and `description`), followed by the steps. Claude only reads the names and descriptions up front, and loads the full body when a request matches. That means a skill costs almost nothing until you actually need it.

| Pattern | Loads | Context cost | Best for |
|---|---|---|---|
| **Skill** | On description match | Low | Task-specific procedures |
| **CLAUDE.md** | Every session | Fixed, always | Always-on project rules |
| **In-context** | Every turn | Grows with the session | One-off, short work |

Skills on the Messages API are **beta** (headers `code-execution-2025-08-25` + `skills-2025-10-02`, running in a code-execution container). **Subagents don't inherit skills**, so list them explicitly. Please verify the headers against the current docs before you build on them.

## Chapter 9 — Multimodal & Batch Ingestion

Here's a cost that catches people off guard: **images cost tokens before Claude reads a single word.** Each 28×28-pixel patch is roughly 1 token, so an image costs about `⌈w/28⌉ × ⌈h/28⌉` tokens. Every model also has a maximum resolution, and larger images are downscaled first. Always work out the cost from your *largest* production image, not your test thumbnails. A resize step is a 10-minute fix before launch and a long one after.

#### What are we going to do?

1. Write a helper that generates a blank white image of any size and packages it as an image block. We only care about size here, not content.
2. Compute the **token estimate from the formula** for three sizes: 280×280, 1000×1000 and 1400×1400.
3. Ask the API what it *actually* counts, and compare the two columns. Where they differ, that's downscaling and limits at work.

In [ ]:
from PIL import Image
import io


def image_block(w, h):
    buf = io.BytesIO()
    Image.new("RGB", (w, h), "white").save(buf, "PNG")
    return {"type": "image", "source": {"type": "base64", "media_type": "image/png",
            "data": base64.standard_b64encode(buf.getvalue()).decode()}}


baseline = n_tokens("Describe this image.")
for w, h in [(280, 280), (1000, 1000), (1400, 1400)]:
    est = math.ceil(w / 28) * math.ceil(h / 28)
    real = client.messages.count_tokens(model=SONNET, messages=[{"role": "user", "content": [
        image_block(w, h), {"type": "text", "text": "Describe this image."}]}]).input_tokens - baseline
    print(f"{w}×{h}: formula ≈ {est:5,} tokens | API reports ≈ {real:5,} (limits/downscaling may differ)")

### How to send an asset

| Method | Overhead | Use when |
|---|---|---|
| **Inline base64** | Full payload on every request | A one-off image |
| **URL** | None, but it must be public, stable and reachable | It's already hosted at a stable URL |
| **Files API** (beta; not on Bedrock/Vertex) | Upload once, then send just a `file_id` | You reuse the same asset, or manage assets separately from inference |

PDFs use a `document` block (same three source types, with an optional `title` and `context`). Everything you've learned about prompting still applies, and images add ambiguity, so tell Claude how to handle it ("if objects overlap, describe each one and note the overlap").

### Batches: the right tool for offline volume

Looping over the sync API **is not batching**. It's just serial calls that still run into rate limits. The **Message Batches API** takes up to 100,000 requests (or 256 MB), runs asynchronously (within 24 hours), costs less per token, and returns results in **arbitrary order**, so you match them up by `custom_id`.

| Use case | API |
|---|---|
| A user is waiting for a classification | Sync |
| A nightly job on 5,000 records | **Batch** |
| Evaluating a new prompt on 2,000 examples | **Batch** |
| A chat reply | Sync |

#### What are we going to do?

Take the same five clauses from Chapter 2 and classify them as a batch:

1. Submit all five as a single batch job, each with a `custom_id` (`clause-0`, `clause-1`, …).
2. Wait, checking the status every 10 seconds, until it ends. This can take a few minutes, so be patient.
3. Collect the results into a dictionary keyed by `custom_id`, then print each label next to its clause. We match by id, **not by position**, because the order isn't guaranteed.

In [ ]:
batch = client.messages.batches.create(requests=[
    {"custom_id": f"clause-{i}",
     "params": {"model": HAIKU, "max_tokens": 20, "system": SYSTEM,
                "messages": [{"role": "user", "content": f"<clause>{c}</clause>"}]}}
    for i, c in enumerate(clauses)
])
print("Submitted batch", batch.id, "— this can take a few minutes…")

while (status := client.messages.batches.retrieve(batch.id)).processing_status != "ended":
    time.sleep(10)

results = {item.custom_id: text_of(item.result.message).strip()
           for item in client.messages.batches.results(batch.id) if item.result.type == "succeeded"}
for i, c in enumerate(clauses):                       # re-join by custom_id, NOT by position
    print(f"{results.get(f'clause-{i}', '?'):12} ← {c[:60]}…")

## Chapter 10 — Debug Challenge

Time to put it all together. The agent below has **four planted bugs**, one in each layer: *schema*, *streaming*, *context* and *memory*. Read the code, hunt for them, and try to find all four **before** you open the answers.

```python
tools = [{
    "name": "get_customer_data",
    "description": "Gets data.",
    "input_schema": {"type": "object", "properties": {"id": {"type": "string"}}, "required": ["id"]},
}]

def run_agent(user_request, session_history):
    messages = session_history + [{"role": "user", "content": user_request}]
    while True:
        blocks, stop_seen = {}, False
        with client.messages.stream(model=model, max_tokens=4096, tools=tools,
                                    messages=messages, thinking={"type": "adaptive"}) as stream:
            for event in stream:
                if event.type == "content_block_start":   blocks[event.index] = init_block(event)
                elif event.type == "content_block_delta": apply_delta(blocks[event.index], event.delta)
                elif event.type == "message_stop":        stop_seen = True
        assistant_content = [b for b in assemble(blocks) if b["type"] != "thinking"]
        messages.append({"role": "assistant", "content": assistant_content})
        response = finalize(blocks)
        if response.stop_reason == "end_turn":
            return response
        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                messages.append({"role": "user", "content": [{"type": "tool_result",
                                 "tool_use_id": block.id, "content": result}]})

def build_session_history(prior_sessions):
    full_history = []
    for session in prior_sessions:
        full_history.extend(session["messages"])      # every prior transcript, in-context
    return full_history
```

<details><summary><b>Show the answers</b></summary>

| Layer | Bug | Runtime effect | Fix |
|---|---|---|---|
| **Schema** | Description is just `"Gets data."` (no when / when-not / returns) | Claude can't route reliably; wrong or missed tool calls | 3–4 sentence description with an exclusion condition |
| **Streaming** | `stop_seen` is set but never checked before `messages.append` | A dropped stream commits a half-built `tool_use` to history → next request fails | Commit only if `stop_seen`; otherwise discard and retry |
| **Context** | Thinking blocks are filtered out of the assistant turn (and each tool result goes in its own user message) | Signature check fails and the request is rejected; multiple `tool_use` results must share **one** user turn | Append `assistant_content` unchanged; collect all results into one user message |
| **Memory** | All prior session transcripts concatenated in-context | Tokens + latency grow every session until the window fills | External storage + summarized/relevant subset at session start |

</details>

## 🏁 Cheat Sheet

You made it! Here's the whole lab in one table, so you have something to come back to.

| Chapter | One-line takeaway |
|---|---|
| Prompting | Diagnose the failure type, add the one missing technique, and use structured outputs when the parser must never break |
| Thinking | On for hard multi-step work, off for lookups; return thinking blocks unchanged |
| Tools | Descriptions drive selection, so say when *not* to use a tool; every `tool_use` needs a matching `tool_result` in the next user turn |
| Streaming | Stream end ≠ message complete; commit only after `message_stop` |
| Context | Measure real tool outputs; prune, compact (with a good summarizer), cache stable prefixes, hand off to subagents |
| Agents | Workflow vs agent first; gate irreversible actions with a human; set explicit exit conditions |
| Memory | Choose the scope at design time; Skills load on demand |
| Multimodal/Batch | Price images before building; batch ≠ looping; join by `custom_id` |

**Next up:** *Claude Code, MCP & Integration*. We'll take this working integration and make it configurable, shareable and safe.